# 🫀 퀘스트 46 · Q8 — **가중 매크로**: 개체를 버리지 않고 정보를 짜낸다

| | **MedKOS / `notebooks/quest46_q8_weighted_macro.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` · 앞선 실험 `ailab-2026-0047`(Q1) |
| 규약 | `SCORING_RULES.md` **R8 · R11 · R11-b · R12** |

## 왜 이 실험인가

Q1 이 남긴 건 **버리기/받기 이분법**이다:

| GMIN | 환자 | 부트 최대 SE | |
|---:|---:|---:|---|
| 2 | **27** | 0.2661 (#50, 양성 3개) | ❌ SE |
| 10 | 12 | 0.048 | ❌ 환자 |

**제3의 길**: 다 받되 **정밀도로 가중**한다. `w_i = 1/SE_i²`(역분산 — 메타분석 표준).
양성 3개짜리는 자동으로 거의 0 가중이 되므로 "버린 것과 비슷하되 정보는 남는다".

## 이 방법의 위험 — 관문에 넣었다

역분산 가중은 **정밀한 소수에게 가중을 몰아줄 수 있다.** 그러면 이름만 매크로지
사실상 한두 명의 지표다 — **R11-b 에서 #213 하나가 매크로의 110% 를 만든 것과 같은
사고**다.

그래서 **Kish 유효표본크기**를 관문으로 쓴다:

```
ESS = (Σw)² / Σw²      모두 같은 가중이면 ESS = n · 한 명에 몰리면 ESS → 1
```

## 사전등록

| 관문 | 내용 |
|---|---|
| **Q8-1** | 가중 매크로의 환자 부트 CI 폭 < 단순 매크로(GMIN 10) |
| **Q8-2** | 커버리지 ≥ 단순 매크로 |
| **Q8-3** | **편향 없음** — 단순 점추정이 가중 CI 안에 있다 |
| **Q8-4** | **Kish ESS ≥ 20** |

⚠️ 역분산 가중은 SE 를 **정확히** 알 때만 최적이다. 부트 SE 자체가 양성이 적을수록
불안정하므로 **가중도 그만큼 불안정하다**(가중의 가중 문제).

⚠️ 이건 **채점 방법**이지 데이터를 늘리는 게 아니다. **ESS 가 진짜 상한**이다.


In [ ]:
# CELL 0 — 공용 사전점검 (pipelines/SCORING_RULES.md)
import numpy as np
from scipy import stats

def decide(lo, hi, thr, direction):
    """사전등록 관문의 유일한 계약: 지지 / 기각 / **미결**."""
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def t_ci(v, conf=0.95):
    v = np.asarray([x for x in v if np.isfinite(x)], float); n = len(v)
    m = float(v.mean()) if n else float("nan")
    if n < 2: return m, np.nan, np.nan
    h = float(stats.t.ppf(.5 + conf / 2, n - 1) * v.std(ddof=1) / np.sqrt(n))
    return m, m - h, m + h

class AssetError(RuntimeError): pass
print("CELL 0 ✅ decide · t_ci 준비")

In [ ]:
# CELL 1 — 설정
import os, sys, json, time, subprocess, importlib
importlib.invalidate_caches()
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

# ★ 학습 0회. 실험22-A 가 남긴 예측 캐시만 읽는다.
CACHE = os.path.join(PROJECT, "data", "exp22a_probs_s5_v1.npz")

SE_CAP   = 0.05     # 환자별 AUROC 의 표집 SE 상한(중앙값)
SE_MULT  = 2.0      # 최대 SE 는 SE_CAP × 이 배수 이내 — #213 같은 시한폭탄 차단
N_MIN    = 20       # 채점 환자 하한 (Q1 합격 기준)
COV_MIN  = 0.70     # 채점 환자가 덮어야 할 양성 비율
DOM_MAX  = 0.50     # 채점 환자 안에서의 지배 지분 상한 (R11-3)
GMINS    = [2, 5, 10, 15, 20, 30, 40, 50, 75, 100, 150, 200]
SEED0    = 20260802

_DS1 = [101,106,108,109,112,114,115,116,118,119,122,124,201,203,205,207,208,209,215,220,223,230]
_DS2 = [100,103,105,111,113,117,121,123,200,202,210,212,213,214,219,221,222,228,231,232,233,234]

CONFIG = dict(
    exp="quest46_q8_weighted_macro", quest="ailab-2026-0046", step="weighted-macro",
    parent_exp=["exp22a_axis_transfer", "ailab-2026-0045"],
    purpose=("Q1 이 남긴 딜레마: GMIN 을 올리면 SE 는 잡히는데 환자가 줄고(12명), "
             "내리면 환자는 느는데(27명) 양성 3개짜리가 섞여 SE 가 0.27 로 터진다. "
             "버리는 대신 **역분산 가중**으로 다 받으면 어떻게 되나. 학습 0회"),
    dataset="MIT-BIH DS2(within · 대조군) + INCART(cross · 본 대상)",
    change_one_thing="모델·확률 그대로. **채점 대상 선정 규칙만** 바꾼다",
    thresholds=dict(se_cap=SE_CAP, se_mult=SE_MULT, n_min=N_MIN,
                    cov_min=COV_MIN, dom_max=DOM_MAX),
    why_se=("`GMIN` 을 **계단 크기**(R-prec 의 1/n_pos)로 정하면 지표마다 답이 달라진다. "
            "AUROC 는 계단이 1/(n_pos·n_neg) 로 촘촘해 n_pos=2 도 통과한다. "
            "그래서 기준을 **표집 SE** 로 잡는다 — 두 지표 모두에서 '이 환자 한 명의 값을 "
            "얼마나 믿을 수 있나' 를 잰다"),
    predictions={
        "Q8-1": "가중 매크로의 환자 부트 CI 폭 < 단순 매크로(GMIN 10)",
        "Q8-2": "가중 매크로의 S 커버리지 ≥ 단순 매크로",
        "Q8-3": "편향 없음 — 단순 점추정이 가중 CI 안에 있다",
        "Q8-4": f"**Kish ESS ≥ {N_MIN}** — 가중이 소수에 몰리면 이름만 매크로다"},
    caveat=("INCART 의 `pid` 는 레코드(75)이고 실제 환자는 32명이다(§6.5 L3). "
            "여기서 '환자' 는 **레코드**를 뜻한다 — 환자 단위 CI 는 그만큼 낙관적이다. "
            "Hanley SE 는 보수적 근사라(부트 대비 1.1~1.6배) 하한 설정용으로만 쓴다"))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q8_wmacro", CONFIG, project=PROJECT)
run.log(f"설정 ✅ N_MIN {N_MIN} · 캐시 {os.path.basename(CACHE)}")

In [ ]:
# CELL 2 — 【G0】 자산·정합성 점검. 가정이 틀리면 여기서 멈춘다
need = {CACHE: "실험22-A 예측 캐시(학습 0회의 근거)",
        os.path.join(MITBIH, "mamba_data.npz"): "MIT-BIH 라벨·pid",
        os.path.join(MITBIH, "incart_data.npz"): "INCART 라벨·pid"}
miss = [p for p in need if not os.path.exists(p)]
if miss:
    raise AssetError(f"자산 없음: {miss}\n  → 실험22-A(CELL 4)를 먼저 돌려 캐시를 만들 것")
run.log("자산 확인 ✅ " + " · ".join(os.path.basename(p) for p in need))

P = dict(np.load(CACHE))
run.log(f"\n캐시 키: {sorted(P)}")
dm = np.load(os.path.join(MITBIH, "mamba_data.npz"))
di = np.load(os.path.join(MITBIH, "incart_data.npz"))
mpid, my = dm["pid"], dm["y"]
ipid, iy = di["pid"], di["y"]
TE = np.isin(mpid, _DS2)

# ★ 라벨을 캐시와 **대조**한다. 길이만 맞는 걸로는 안 된다(R10-b).
for nm, ycache, ylocal in (("within", P["y_within"], my[TE]), ("cross", P["y_cross"], iy)):
    a, b = np.asarray(ycache), np.asarray(ylocal)
    if a.shape != b.shape:
        raise AssetError(f"{nm} 라벨 길이 불일치 캐시 {a.shape} vs 로컬 {b.shape}")
    if not (a == b).all():
        raise AssetError(f"{nm} 라벨 **내용**이 다르다 — 캐시가 다른 실행의 것이다")
    h = np.bincount(a.astype(int), minlength=3)[:3]
    run.log(f"  {nm:<7} {len(a):>8,}비트 · N/S/V {h.tolist()} · 캐시=로컬 ✅")

COH = {"within (MIT-BIH DS2)": (np.asarray(P["y_within"]), np.asarray(mpid[TE])),
       "cross (INCART)":       (np.asarray(P["y_cross"]),  np.asarray(ipid))}
run.log(f"\n  레코드 수 — within {len(np.unique(mpid[TE]))} · cross {len(np.unique(ipid))}")
run.log("  ⚠️ INCART 의 '레코드' 75개는 실제 환자 32명이다(§6.5 L3). 아래 '환자' 는 레코드다")
run.save_json("config", CONFIG)

In [ ]:
# CELL 2 — 【Q8】 가중 매크로 — 개체를 **버리는 대신 가중**한다
#
#  Q1 이 남긴 딜레마: GMIN 을 올리면 SE 는 잡히는데 환자가 줄고(12명), 내리면 환자는
#  느는데(27명) 양성 3개짜리가 섞여 SE 가 0.27 로 터진다. **버리기/받기 이분법**이다.
#
#  제3의 길: 다 받되 **정밀도로 가중**한다. w_i = 1/SE_i²(역분산 가중 — 메타분석 표준).
#  양성 3개짜리는 자동으로 거의 0 가중이 되므로 '버린 것과 비슷하되 정보는 남는다'.
#
#  ★ 위험 1 — **추정 잡음**: 우리가 쓰는 SE 는 참값이 아니라 부트 추정치다. SE 가
#    코호트 안에서 **균질**하면 1/SE² 는 '우연히 SE 가 작게 나온 개체' 에 가중을
#    몰아주고, 그건 순수한 잡음이다. 픽스처 실측: 25개체가 전부 같은 조건일 때
#    가중 CI 폭 0.0425 vs 단순 0.0254 — **가중이 1.7배 나빴다.** ESS 도 25 → 18.
#    → 가중은 SE 가 **이질적일 때만** 이득이다. 그래서 아래에 잡음이 없는 대안
#      **양성수 가중**(w = n_pos)을 나란히 놓고 비교한다.
#
#  ★ 위험 2: 역분산 가중은 **정밀한 소수에게 가중을 몰아줄 수 있다.** 그러면 이름만
#    매크로지 사실상 한두 명의 지표가 된다 — R11-b 에서 #213 하나가 매크로의 110% 를
#    만든 것과 **같은 사고**다. 그래서 **Kish 유효표본크기**를 관문에 넣는다:
#        ESS = (Σw)² / Σw²      (모두 같은 가중이면 ESS = n, 한 명에 몰리면 ESS → 1)
#    ESS 가 N_MIN 미만이면 가중 매크로도 '환자를 확보했다' 고 말할 수 없다.
from sklearn.metrics import roc_auc_score

NB_BOOT_Q8 = 400
IDX_S = 1

def per_record(prob, y, g, recs, idx, nboot):
    """레코드별 (AUROC, 부트 SE, 양성 수). 가중치는 SE 에서 나온다."""
    rng = np.random.RandomState(SEED0)
    out = {}
    for r in recs:
        m = np.where(g == r)[0]
        t = (y[m] == idx); s = prob[m]
        if not (t.any() and not t.all()):
            continue
        a = roc_auc_score(t.astype(int), s)
        vals = []
        for _ in range(nboot):
            j = rng.randint(0, len(m), len(m)); tj = t[j]
            if 0 < tj.sum() < len(tj):
                vals.append(roc_auc_score(tj.astype(int), s[j]))
        se = float(np.std(vals, ddof=1)) if len(vals) > 2 else np.nan
        out[int(r)] = (float(a), se, int(t.sum()))
    return out

def kish_ess(w):
    """Kish 유효표본크기. 가중이 한 명에 몰리면 1 로 간다."""
    w = np.asarray(w, float)
    return float(w.sum() ** 2 / max((w ** 2).sum(), 1e-30))

def wmean(a, w):
    w = np.asarray(w, float); a = np.asarray(a, float)
    return float((a * w).sum() / w.sum())

run.log("\n" + "=" * 100)
run.log("【Q8】 가중 매크로 vs 단순 매크로 — 개체를 버리지 않고 정보를 짜낸다")
run.log("=" * 100)
Y, G = np.asarray(P["y_cross"]), np.asarray(ipid)
SC = P["v2_cross_raw"].mean(0)[:, IDX_S]
ALLREC = [int(r) for r in np.unique(G)]
PR = per_record(SC, Y, G, ALLREC, IDX_S, NB_BOOT_Q8)
run.log(f"  INCART 레코드 {len(ALLREC)}개 중 채점 가능 {len(PR)}개"
        f" (양성·음성이 모두 있는 곳)")

ARMS = {}
for nm, gmin, mode in (("단순 매크로 (GMIN 10)", 10, "plain"),
                       ("단순 매크로 (GMIN 2)", 2, "plain"),
                       ("양성수 가중 (GMIN 2)", 2, "npos"),
                       ("**역분산 가중 (GMIN 2)**", 2, "invvar")):
    # ★ `se > 0` 은 **역분산 arm 에만** 건다. 0 나눗셈을 막으려고 넣은 조건인데
    #   단순 매크로에까지 걸면 **완벽 분리 개체(SE=0)를 채점에서 떨궈낸다** — 가장
    #   쉬운 개체를 조용히 빼는 것이라 단순 arm 의 값이 왜곡된다(실측 기각 사유 아님,
    #   Q8 로그 ailab-2026-0050 '한계' 참조).
    keep = [r for r, (a, se, p) in PR.items() if p >= gmin and np.isfinite(se)]
    if mode == "invvar":
        drop = [r for r in keep if PR[r][1] <= 0]
        keep = [r for r in keep if PR[r][1] > 0]
        if drop:
            run.log(f"  ⚠️ '{nm}' — SE=0(완벽 분리) 개체 {len(drop)}개 제외: {drop[:6]}"
                    f"  ← 역분산 가중은 이들에게 무한 가중을 준다(R14-b)")
    if len(keep) < 2:
        # 개체가 1개면 '매크로' 가 아니다. 환자 부트 CI 폭이 0 으로 나와 **가짜 정밀도**가
        # 된다(리샘플해도 늘 같은 한 명). 비교 대상에서 제외하고 그 사실을 남긴다.
        run.log(f"  ⚠️ '{nm}' 제외 — 조건을 만족하는 개체가 {len(keep)}개뿐이다"
                f" (매크로 성립 불가)")
        continue
    a = np.array([PR[r][0] for r in keep])
    se = np.array([PR[r][1] for r in keep])
    pos = np.array([PR[r][2] for r in keep])
    # 세 가중: 균등 · 양성수(추정 잡음 없음) · 역분산(최적이지만 잡음에 취약)
    #  ⚠️ `npos` 가중의 최대 가중은 정의상 **지배 지분**과 같다
    #     (max(n_pos)/Σn_pos). 즉 양성수 가중은 매크로를 **전역 지표 쪽으로 되감는다**
    #     — R11 이 피하려던 바로 그 방향이다. 실측: 최대가중 30.2% = INCART 지배 30.1%.
    w = {"plain": np.ones(len(keep)),
         "npos": pos.astype(float),
         "invvar": 1.0 / np.maximum(se, 1e-12) ** 2}[mode]
    ARMS[nm] = dict(keep=keep, a=a, se=se, pos=pos, w=w, gmin=gmin, mode=mode,
                    est=wmean(a, w), ess=kish_ess(w),
                    cov=float(pos.sum() / max(sum(p for _, _, p in PR.values()), 1)),
                    top_w=float(np.max(w) / w.sum()))

In [ ]:
# CELL 3 — 【Q8 채점】 CI 폭 · ESS · 편향
run.log(f"\n  {'구성':<26}{'개체':>5}{'ESS':>7}{'최대가중':>9}{'커버':>8}"
        f"{'점추정':>9}{'환자 부트 95% CI':>24}{'폭':>8}")
rng = np.random.RandomState(SEED0 + 1)
RES = {}
for nm, d in ARMS.items():
    bs = []
    for _ in range(4000):
        i = rng.randint(0, len(d["a"]), len(d["a"]))
        bs.append(wmean(d["a"][i], d["w"][i]))
    lo, hi = np.percentile(bs, [2.5, 97.5])
    RES[nm] = dict(est=d["est"], lo=float(lo), hi=float(hi), width=float(hi - lo),
                   ess=d["ess"], n=len(d["a"]), cov=d["cov"], top_w=d["top_w"])
    run.log(f"  {nm:<26}{len(d['a']):>5}{d['ess']:>7.1f}{d['top_w']:>9.1%}"
            f"{d['cov']:>8.1%}{d['est']:>9.4f}"
            f"   [{lo:.4f}, {hi:.4f}]{hi-lo:>8.4f}")

base = RES.get("단순 매크로 (GMIN 10)")
wgt = RES.get("**역분산 가중 (GMIN 2)**")
nps = RES.get("양성수 가중 (GMIN 2)")
VERD = {}
def g_(k, ok, d):
    VERD[k] = "✅ 지지" if ok else "❌ 기각"
    run.log(f"  {k:<7}{VERD[k]}  {d}")

run.log("")
if base and wgt:
    _rel = (f"  ({(1 - wgt['width'] / base['width']) * 100:+.1f}%)"
            if base["width"] > 0 else "  (기준 폭 0 — 상대비교 생략)")
    g_("Q8-1", wgt["width"] < base["width"],
       f"CI 폭 가중 **{wgt['width']:.4f}** vs 단순(GMIN10) {base['width']:.4f}" + _rel)
    g_("Q8-2", wgt["cov"] >= base["cov"],
       f"커버리지 가중 **{wgt['cov']:.1%}** ≥ 단순 {base['cov']:.1%}")
    g_("Q8-3", wgt["lo"] <= base["est"] <= wgt["hi"],
       f"편향 점검 — 단순 점추정 {base['est']:.4f} 가 가중 CI "
       f"[{wgt['lo']:.4f}, {wgt['hi']:.4f}] 안에 있나")
    if nps:
        run.log(f"\n  참고 — 양성수 가중(추정 잡음 없음): CI 폭 {nps['width']:.4f}"
                f" · ESS {nps['ess']:.1f} · 점추정 {nps['est']:.4f}")
        if nps["width"] < wgt["width"]:
            run.log("    → **양성수 가중이 역분산보다 낫다.** SE 추정 잡음이 역분산을"
                    " 갉아먹고 있다는 뜻이다 — 이 코호트에서는 역분산을 쓰지 않는다.")
    g_("Q8-4", wgt["ess"] >= N_MIN,
       f"**Kish ESS {wgt['ess']:.1f}** ≥ {N_MIN}  (최대 가중 {wgt['top_w']:.1%})"
       f"  ← 가중이 소수에 몰리면 이름만 매크로다")
    run.log("\n  " + "  ".join(f"{k}: {v}" for k, v in VERD.items()))
    if VERD.get("Q8-4", "").startswith("❌"):
        # ★ 관문에는 **읽는 순서**가 있다. ESS 가 무너지면 Q8-1 의 '좁은 CI' 는
        #   정밀해진 게 아니라 **개체가 한 줌으로 붕괴한** 결과다. 픽스처 실측:
        #   양성 3,000개 1개 + 12개짜리 26개 코호트에서 Q8-1 은 '지지'(폭 46.6% 감소)
        #   인데 ESS 는 1.8 이었다. 폭만 보면 이겼다고 쓸 뻔했다.
        run.log(f"  ⚠️ ESS 가 {wgt['ess']:.1f} 다 — 개체 {wgt['n']}개를 받았지만 실질은"
                f" 그보다 훨씬 적다. R11-b 의 #213 과 같은 구조다.")
        run.log("  ⛔ **Q8-4 가 기각이면 Q8-1·Q8-3 은 읽지 않는다.** 좁아진 CI 는"
                " 정밀해져서가 아니라 유효 개체가 무너져서 생긴 것이다.")
        for k in ("Q8-1", "Q8-3"):
            if VERD.get(k, "").startswith("✅"):
                VERD[k] += " (ESS 붕괴로 판독 보류)"
else:
    # 관문을 **조용히 비우지 않는다**. 왜 못 쟀는지가 결과다.
    for k in ("Q8-1", "Q8-2", "Q8-3", "Q8-4"):
        VERD[k] = "⚠️ 미결"
    if wgt:
        # 가중 arm 은 섰는데 기준(GMIN 10)이 안 선 경우 — ESS 만은 잴 수 있다
        g_("Q8-4", wgt["ess"] >= N_MIN,
           f"**Kish ESS {wgt['ess']:.1f}** ≥ {N_MIN}  (최대 가중 {wgt['top_w']:.1%})")
    miss = [k for k, v in (("단순 매크로 (GMIN 10)", base),
                           ("**역분산 가중 (GMIN 2)**", wgt)) if not v]
    run.log(f"  ⚠️ 비교 불가 — 성립하지 않은 arm: {miss}."
            f"  GMIN 10 을 넘는 개체가 2개 미만이면 기준선 자체가 없다")

run.log("\n  ⚠️ 역분산 가중은 SE 를 **정확히** 알 때만 최적이다. 부트 SE 자체가 양성이")
run.log("     적을수록 불안정하므로, 가중도 그만큼 불안정하다(가중의 가중 문제).")
run.log("  ⚠️ 이건 채점 방법이지 데이터를 늘리는 게 아니다. ESS 가 진짜 상한이다.")
CONFIG["result"] = {"verdicts": VERD, "arms": RES,
                    "per_record": {r: {"auroc": v[0], "se": v[1], "pos": v[2]}
                                   for r, v in sorted(PR.items())}}
run.save_json("config", CONFIG)